In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import re
import time
from datetime import datetime
import credentials

In [2]:
conn_str = 'mysql+mysqlconnector://' + \
	f"{credentials.mysql['username']}:{credentials.mysql['password']}" + \
	'@localhost:3306/Mutis'
engine = create_engine(conn_str)
connection = engine.connect()

In [3]:
colls = pd.read_sql("SELECT OccurrenceID, DB2015ID, CollectorVerbatim, CollectionNumberVerbatim FROM Occurrences", connection)
colls[["SpecimenCode", "Institution"]] = None, None

In [4]:
colpp = {
	'Colector depurado': 'colde', 'Colector principal': 'colpri', 
	'Número de colector': 'CollectionNumberVerbatim', 
	'Fecha_Inicial_(Year)': 'year',
	'Fecha_Inicial_(Month)': 'month', 'Fecha_Inicial_(day)': 'day',
	'Notas_Localidad_y_Habitat': 'notas', 'Fenologia': 'feno',
	'Usos': 'usos', 'Nombres_comunes': 'comname', 
	'Comentarios generales': 'comm', 
	'Fuente': 'fuente', 'Referencia bibligráfica': 'ref',
	'Herbario de proveniencia': 'herb_spec','Origen para herbario':'herb_doc',
	'Numero original en Base 2015': 'dbid', 
	'Nombre base para flora de Bogota': 'name',
	'Descripcion ejemplar': 'descr', 'Localidad': 'Localidad',
	'Latitud_decimal': 'Latitud_decimal', 'Longitud_decimal': 'Longitud_decimal',
	'Código de localidad georreferenciada': 'DB2020ID',
	'Latitud_original': 'Latitude_text', 'Longitud_original': 'Longitude_text',
	'Latitud para trabajo de mapas':'latmaps', 'Longitud para trabajo de mapas': 'lonmaps',
	'Extensión de georreferenciación': 'Uncertainty', 'DATUM': 'Datum',
	'Elev__minima': 'ElevationMin', 'Elev__maxima': 'ElevationMax',
	'State': 'State', 'County': 'County',
	'Nombre base para flora de Bogota': 'nombre',
	'Determinador en fuente': 'deterFuente',
	'Determinador Depurado' : 'deter',
	'Calificador en fuente' : 'deterUnc',
	'Fecha_Determinacion en fuente': 'deterFecha',
	'Fecha_Determinacion_(Year)': 'deterA',
	'Fecha_Determinacion_(Month)' : 'deterM',
	'Fecha_Determinacion_(Day)' : 'deterD',
	"Origen para herbario": "herbarium", "Número de identificación": "specimenNumber"
	}

In [5]:
orifile = "../../JBB/Datos/Flora_de_Bogota/Registros_2020/FlBogota2020.csv"
colpp = {
	'Colector depurado': 'colde', 'Colector principal': 'colpri', 
	'Número de colector': 'CollectionNumberVerbatim', 
	'Numero original en Base 2015': 'dbid', 
	"Origen para herbario": "herbarium", "Número de identificación": "specimenNumber"
	}

ori = pd.read_csv(orifile, dtype={'Elev__minima': str, 'Elev__maxima': str},
	usecols=colpp)

ori = ori.rename(columns=colpp)

In [6]:
ori.loc[ori.colde.isna(), 'colde'] = ori.loc[ori.colde.isna(), 'colpri']

bits = ori.loc[ori.CollectionNumberVerbatim.notna()
	& ori.CollectionNumberVerbatim.str.contains(r'[^\d]'), 'CollectionNumberVerbatim'
	].str.extractall(r'(\d+)'
	).reset_index(
	).pivot(index='level_0', columns='match', values=0
	)

ori = ori.merge(bits, 'left', left_index=True, right_index=True
	)

ori['CollectionNumber'] = np.nan

# Sorting number before spliting token
ori.loc[ori.CollectionNumberVerbatim.notna() & ori[1].notna()
	& ori.CollectionNumberVerbatim.str.contains(r'\d+[^\d]+\d{1,2}([^\d]+$|$)'),
	'CollectionNumber'] = \
	ori.loc[ori.CollectionNumberVerbatim.notna() & ori[1].notna()
		& ori.CollectionNumberVerbatim.str.contains(r'\d+[^\d]+\d{1,2}([^\d]+$|$)'), 
		0].astype(np.int64)

# Sorting number after spliting token
ori.loc[ori.CollectionNumberVerbatim.notna() & ori[1].notna()
	& ori.CollectionNumberVerbatim.str.contains(r'[^\d]\d{3,}'),
	'CollectionNumber'] = \
	ori.loc[ori.CollectionNumberVerbatim.notna() & ori[1].notna()
		& ori.CollectionNumberVerbatim.str.contains(r'[^\d]\d{3,}'), 
		1].astype(np.int64)

# Remove non-digits from collections numbers
ori.loc[ori.CollectionNumber.isna() & 
	ori.CollectionNumberVerbatim.str.contains(r'(^[^\d]+\d+$|^\d+[^\d]+$)'), 
	'CollectionNumber'
	] = ori.loc[ori.CollectionNumber.isna() & 
	ori.CollectionNumberVerbatim.str.contains(r'(^[^\d]+\d+$|^\d+[^\d]+$)'), 
	'CollectionNumberVerbatim'
	].str.replace(r'[^\d]+', '', regex=True).astype(np.int64)

# Remove non-digits from collections numbers (again)
ori.loc[ori.CollectionNumber.isna() & 
	ori.CollectionNumberVerbatim.str.contains(r'[^\d]'), 
	'CollectionNumber'] = ori.loc[ori.CollectionNumber.isna() & 
	ori.CollectionNumberVerbatim.str.contains(r'[^\d]'), 
	'CollectionNumberVerbatim'].str.replace(r'[^\d]', '', regex=True
	).astype(np.int64)

# Sorting number is verbatim number
ori.loc[ori.CollectionNumber.isna() & ori.CollectionNumberVerbatim.notna(), 'CollectionNumber'] = ori.loc[ori.CollectionNumber.isna() & ori.CollectionNumberVerbatim.notna(), 'CollectionNumberVerbatim'].astype(np.int64)


/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/222228883.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & ori.CollectionNumberVerbatim.str.contains(r'\d+[^\d]+\d{1,2}([^\d]+$|$)'),
/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/222228883.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & ori.CollectionNumberVerbatim.str.contains(r'\d+[^\d]+\d{1,2}([^\d]+$|$)'),
/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/222228883.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ori.CollectionNumberVerbatim.str.contains(r'(^[^\d]+\d+$|^\d+[^\d]+$)'),
/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/222228883.py:33: UserWarning: This pattern is interpreted

In [9]:
ori[ori.dbid.notna()].shape, colls[colls.DB2015ID.notna()].shape

((26643, 10), (26643, 6))

In [ ]:
for row in colls.itertuples():
	if pd.notna(row.DB2015ID):
		#pass
		th = ori.loc[ori.dbid == row.DB2015ID]
		if th.shape[0] == 1:
			colls.loc[colls.DB2015ID == row.DB2015ID, "Institution"] = th.herbarium.item()
			colls.loc[colls.DB2015ID == row.DB2015ID, "SpecimenCode"] = th.specimenNumber.item()
		elif th.shape[0] >= 2:
			if th.dbid.unique().shape[0] == 1:
				#print(th.iloc[0])
				colls.loc[colls.DB2015ID == row.DB2015ID, "Institution"] = th.herbarium.iloc[0]
				colls.loc[colls.DB2015ID == row.DB2015ID, "SpecimenCode"] = th.specimenNumber.iloc[0]
			else:
				raise ValueError("Multiple DB2020IDs for a single occurrence.")
		else:
			raise ValueError("Mutis entry has no DB2020 entry.")

In [ ]:
for un in colls.loc[
		colls.Institution.isna() &
		colls.CollectorVerbatim.notna() &
		colls.CollectionNumberVerbatim.notna()
	].groupby(["CollectorVerbatim", "CollectionNumberVerbatim"]
	).size(
	).reset_index(
	).itertuples():

	th = ori.loc[
		(ori.colde == un.CollectorVerbatim) & 
		(ori.CollectionNumberVerbatim == un.CollectionNumberVerbatim)
	]

	qu = (colls.CollectorVerbatim == un.CollectorVerbatim) & \
		(colls.CollectionNumberVerbatim == un.CollectionNumberVerbatim)

	if th.shape[0] == 1:
		colls.loc[qu, "Institution"] = th.herbarium.item()
		colls.loc[qu, "SpecimenCode"] = th.specimenNumber.item()
	
	elif th.shape[0] == colls[qu].shape[0]:
		colls.loc[qu, "Institution"] = th.herbarium
		colls.loc[qu, "SpecimenCode"] = th.specimenNumber
	
	else:
		raise ValueError(f"Mapped specimen data does not match colls size: {th.shape=}, {colls[qu].shape=}")
		#print(th.shape, colls[qu].shape)


In [12]:
colls["Institution"] = colls.Institution.str.upper()
colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+\d+$", na=False), 
	"SpecimenCode"] = \
colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+\d+$", 
	na=False)].Institution.str.extract(r"SCHNEIDER(\s+NO\.)?\s+(\d+)$")[1].values
colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+(\d+)$", na=False), 
	"Institution"] = "HERBARIO SCHNEIDER"

/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/1167308949.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+\d+$",
/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/1167308949.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+\d+$", na=False),
/var/folders/r9/h_r0bsyx5cs48jl3mmb_0gyh0000gn/T/ipykernel_73240/1167308949.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  colls.loc[colls.Institution.str.contains(r"SCHNEIDER(\s+NO\.)?\s+(\d+)$", na=False),


In [13]:
def splitherb(ath):
	if isinstance(ath, str):
		return re.split(r",\s*", ath)
	else:
		return [None]

In [14]:
colls["multiherb"] = colls.Institution.apply(splitherb)
colls = colls.explode("multiherb")
#colls["Institution"] = colls.multiherb

In [19]:
instis = colls.groupby("multiherb"
).size(
).reset_index()

In [20]:
instis["InstitutionID"] = instis.index + 1
instis["Name"] = None
instis["Code"] = None
instis["TimeStamp"] = datetime.strptime("2020-01-01", "%Y-%m-%d")

In [23]:
instis.loc[instis.multiherb.str.len() > 9, "Name"] = instis.loc[instis.multiherb.str.len() > 9, "multiherb"]
instis.loc[instis.multiherb.str.len() <= 9, "Code"] = instis.loc[instis.multiherb.str.len() <= 9, "multiherb"]

In [37]:
colls = colls.merge(
	instis[["multiherb", "InstitutionID"]],
	how="left",
	on="multiherb",
)

In [40]:
instis.drop(
	columns=[0, "multiherb"]
).to_sql('Institutions', engine, if_exists='append', index=False, 
	chunksize=10000, method='multi'
)

80

In [43]:
colls["File"] = None
colls["TimeStamp"] = datetime.strptime("2020-01-01", "%Y-%m-%d")

In [53]:
colls = colls.reset_index(drop=True)
colls = colls.drop(index=colls[colls.Institution.isna() & colls.SpecimenCode.isna()].index
	).reset_index(drop=True)

In [54]:
colls["SpecimenID"] = colls.index + 1

In [59]:
colls[['SpecimenID', 'OccurrenceID', 'SpecimenCode', 'InstitutionID', 'File','TimeStamp']
	].rename(
		columns={"OccurrenceID": "Occurrence", "InstitutionID": "Institution"}
	).to_sql('Specimens', engine, if_exists='append', index=False, 
		chunksize=10000, method='multi'
	)

34858

In [60]:
connection.close()